In [ ]:
import tensorflow as tf

In [ ]:
import numpy as np
from tensorflow import keras
import pandas as pd
from keras import layers

In [ ]:
import numpy as np
import gc

FILE_PATH = 'dsprites_dataset_filepath'

with np.load(FILE_PATH, allow_pickle=True) as data:
    imgs_full = data['imgs']
    latents_classes = data['latents_classes']

max_pixel = imgs_full.max()

N_TOTAL = 20000
total_size = len(imgs_full)
idx = np.linspace(0, total_size - 1, N_TOTAL, dtype=int)

np.random.RandomState(42).shuffle(idx)

x_subset = imgs_full[idx]
factors_subset = latents_classes[idx]

del imgs_full, latents_classes, data
gc.collect()


if max_pixel > 1.0:
    x_subset = x_subset.astype("float32") / 255.0
else:
    x_subset = x_subset.astype("float32")

x_subset = np.expand_dims(x_subset, axis=-1)

x_train = x_subset[:18000]
x_test = x_subset[18000:]
factors_test = factors_subset[18000:]

del x_subset, factors_subset
gc.collect()



In [ ]:
import gc
import keras
import pandas as pd
import numpy as np
import tensorflow as tf

#


inputs = keras.Input(shape=(64, 64, 1))
x = layers.Flatten()(inputs)
h1 = layers.Dense(64, activation="relu")(x)
h2 = layers.Dense(32, activation="relu")(h1)
z = layers.Dense(10)(h2)
d1 = layers.Dense(32, activation="relu")(z)
d2 = layers.Dense(64, activation="relu")(d1)
d_out = layers.Dense(64*64*1, activation="sigmoid")(d2)
outputs = layers.Reshape((64, 64, 1))(d_out)

model = keras.Model(inputs, outputs)
model.compile(optimizer="adam", loss="mse")

history = model.fit(
        x_train, x_train,
        validation_data=(x_test, x_test),
        epochs=50,
        batch_size=32,
        verbose=1
)

encoder = keras.Model(inputs, z)
latents_pred = encoder.predict(x_test, verbose=1)
true_factors = factors_test

    

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

MODEL_NAME = "Small_No_Res"    

shape_labels = factors_test[:1000, 1]

square_idx  = np.where(shape_labels == 0)[0][0]
ellipse_idx = np.where(shape_labels == 1)[0][0]
heart_idx   = np.where(shape_labels == 2)[0][0]

sample_idx = [square_idx, ellipse_idx, heart_idx]

sample_images = x_test[sample_idx]
reconstructions = model.predict(sample_images, verbose=0)

fig, axes = plt.subplots(3, 2, figsize=(6, 9))

for i in range(3):
    axes[i, 0].imshow(sample_images[i].squeeze(), cmap='gray', vmin=0, vmax=1)
    axes[i, 0].axis('off')
    axes[i, 1].imshow(reconstructions[i].squeeze(), cmap='gray', vmin=0, vmax=1)
    axes[i, 1].axis('off')
    if i == 0:
        axes[i, 0].set_title("Original", fontsize=12)
        axes[i, 1].set_title("Reconstruction", fontsize=12)

plt.suptitle(f"{MODEL_NAME} on dSprites", fontsize=14)
plt.tight_layout()
plt.savefig(f"recon_{MODEL_NAME}.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
inputs = keras.Input(shape=(64, 64, 1))
x = layers.Flatten()(inputs)
    
h1 = layers.Dense(1024, activation="relu")(x)
h2 = layers.Dense(512, activation="relu")(h1)
h3 = layers.Dense(256, activation="relu")(h2)
h4 = layers.Dense(128, activation="relu")(h3)
h5 = layers.Dense(64, activation="relu")(h4)
z = layers.Dense(10)(h5)  

d1 = layers.Dense(64, activation="relu")(z)
d2 = layers.Dense(128, activation="relu")(d1)
d3 = layers.Dense(256, activation="relu")(d2)
d4 = layers.Dense(512, activation="relu")(d3)
d5 = layers.Dense(1024, activation="relu")(d4)
d_out = layers.Dense(64*64*1, activation="sigmoid")(d5)
outputs = layers.Reshape((64, 64, 1))(d_out)


model = keras.Model(inputs, outputs)
model.compile(optimizer="adam", loss="mse")

history = model.fit(
        x_train, x_train,
        validation_data=(x_test, x_test),
        epochs=50,
        batch_size=32,
        verbose=1
    )

encoder = keras.Model(inputs, z)
latents_pred = encoder.predict(x_test, verbose=0)
true_factors = factors_test

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

MODEL_NAME = "Big_No_Res"    


shape_labels = factors_test[:1000, 1]

square_idx  = np.where(shape_labels == 0)[0][0]
ellipse_idx = np.where(shape_labels == 1)[0][0]
heart_idx   = np.where(shape_labels == 2)[0][0]

sample_idx = [square_idx, ellipse_idx, heart_idx]

sample_images = x_test[sample_idx]
reconstructions = model.predict(sample_images, verbose=0)

fig, axes = plt.subplots(3, 2, figsize=(6, 9))

for i in range(3):
    axes[i, 0].imshow(sample_images[i].squeeze(), cmap='gray', vmin=0, vmax=1)
    axes[i, 0].axis('off')
    axes[i, 1].imshow(reconstructions[i].squeeze(), cmap='gray', vmin=0, vmax=1)
    axes[i, 1].axis('off')
    if i == 0:
        axes[i, 0].set_title("Original", fontsize=12)
        axes[i, 1].set_title("Reconstruction", fontsize=12)

plt.suptitle(f"{MODEL_NAME} on dSprites", fontsize=14)
plt.tight_layout()
plt.savefig(f"recon_{MODEL_NAME}.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:

inputs = keras.Input(shape=(64, 64, 1))
x = layers.Flatten()(inputs)

h1 = layers.Dense(64)(x)
shortcut1 = layers.Dense(64)(x)
h1 = layers.Add()([h1, shortcut1])
h1 = layers.ReLU()(h1)
h2 = layers.Dense(32)(h1)
shortcut2 = layers.Dense(32)(h1)
h2 = layers.Add()([h2, shortcut2])
h2 = layers.ReLU()(h2)

z = layers.Dense(10)(h2)  

d1 = layers.Dense(32)(z)
shortcut_d1 = layers.Dense(32)(z)
d1 = layers.Add()([d1, shortcut_d1])
d1 = layers.ReLU()(d1)

d2 = layers.Dense(64)(d1)
shortcut_d2 = layers.Dense(64)(d1)
d2 = layers.Add()([d2, shortcut_d2])
d2 = layers.ReLU()(d2)

d_out = layers.Dense(64*64*1, activation="sigmoid")(d2)
outputs = layers.Reshape((64, 64, 1))(d_out)


model = keras.Model(inputs, outputs)
model.compile(optimizer="adam", loss="mse")

history = model.fit(
        x_train, x_train,
        validation_data=(x_test, x_test),
        epochs=50,
        batch_size=32,
        verbose=1
)

encoder = keras.Model(inputs, z)
latents_pred = encoder.predict(x_test, verbose=0)
true_factors = factors_test

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

MODEL_NAME = "Small_Res"   


shape_labels = factors_test[:1000, 1]

square_idx  = np.where(shape_labels == 0)[0][0]
ellipse_idx = np.where(shape_labels == 1)[0][0]
heart_idx   = np.where(shape_labels == 2)[0][0]

sample_idx = [square_idx, ellipse_idx, heart_idx]

sample_images = x_test[sample_idx]
reconstructions = model.predict(sample_images, verbose=0)

fig, axes = plt.subplots(3, 2, figsize=(6, 9))

for i in range(3):
    axes[i, 0].imshow(sample_images[i].squeeze(), cmap='gray', vmin=0, vmax=1)
    axes[i, 0].axis('off')
    axes[i, 1].imshow(reconstructions[i].squeeze(), cmap='gray', vmin=0, vmax=1)
    axes[i, 1].axis('off')
    if i == 0:
        axes[i, 0].set_title("Original", fontsize=12)
        axes[i, 1].set_title("Reconstruction", fontsize=12)

plt.suptitle(f"{MODEL_NAME} on dSprites", fontsize=14)
plt.tight_layout()
plt.savefig(f"recon_{MODEL_NAME}.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:





inputs = keras.Input(shape=(64, 64, 1))
x = layers.Flatten()(inputs)

h1 = layers.Dense(1024)(x)
shortcut1 = layers.Dense(1024)(x)
h1 = layers.Add()([h1, shortcut1])
h1 = layers.ReLU()(h1)
    
h2 = layers.Dense(512)(h1)
shortcut2 = layers.Dense(512)(h1)
h2 = layers.Add()([h2, shortcut2])
h2 = layers.ReLU()(h2)
    
h3 = layers.Dense(256)(h2)
shortcut3 = layers.Dense(256)(h2)
h3 = layers.Add()([h3, shortcut3])
h3 = layers.ReLU()(h3)
    
h4 = layers.Dense(128)(h3)
shortcut4 = layers.Dense(128)(h3)
h4 = layers.Add()([h4, shortcut4])
h4 = layers.ReLU()(h4)
    
h5 = layers.Dense(64)(h4)
shortcut5 = layers.Dense(64)(h4)
h5 = layers.Add()([h5, shortcut5])
h5 = layers.ReLU()(h5)
    
z = layers.Dense(10)(h5)
    
d1 = layers.Dense(64)(z)
shortcut_d1 = layers.Dense(64)(z)
d1 = layers.Add()([d1, shortcut_d1])
d1 = layers.ReLU()(d1)
    
d2 = layers.Dense(128)(d1)
shortcut_d2 = layers.Dense(128)(d1)
d2 = layers.Add()([d2, shortcut_d2])
d2 = layers.ReLU()(d2)
    
d3 = layers.Dense(256)(d2)
shortcut_d3 = layers.Dense(256)(d2)
d3 = layers.Add()([d3, shortcut_d3])
d3 = layers.ReLU()(d3)
    
d4 = layers.Dense(512)(d3)
shortcut_d4 = layers.Dense(512)(d3)
d4 = layers.Add()([d4, shortcut_d4])
d4 = layers.ReLU()(d4)
    
d5 = layers.Dense(1024)(d4)
shortcut_d5 = layers.Dense(1024)(d4)
d5 = layers.Add()([d5, shortcut_d5])
d5 = layers.ReLU()(d5)

d_out = layers.Dense(64*64*1, activation="sigmoid")(d5)
outputs = layers.Reshape((64, 64, 1))(d_out)



model = keras.Model(inputs, outputs)
model.compile(optimizer="adam", loss="mse")

history = model.fit(
        x_train, x_train,
        validation_data=(x_test, x_test),
        epochs=50,
        batch_size=32,
        verbose=1
)

encoder = keras.Model(inputs, z)
latents_pred = encoder.predict(x_test, verbose=0)
true_factors = factors_test


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

MODEL_NAME = "Big_Res"   

shape_labels = factors_test[:1000, 1]

square_idx  = np.where(shape_labels == 0)[0][0]
ellipse_idx = np.where(shape_labels == 1)[0][0]
heart_idx   = np.where(shape_labels == 2)[0][0]

sample_idx = [square_idx, ellipse_idx, heart_idx]

sample_images = x_test[sample_idx]
reconstructions = model.predict(sample_images, verbose=0)

fig, axes = plt.subplots(3, 2, figsize=(6, 9))

for i in range(3):
    axes[i, 0].imshow(sample_images[i].squeeze(), cmap='gray', vmin=0, vmax=1)
    axes[i, 0].axis('off')
    axes[i, 1].imshow(reconstructions[i].squeeze(), cmap='gray', vmin=0, vmax=1)
    axes[i, 1].axis('off')
    if i == 0:
        axes[i, 0].set_title("Original", fontsize=12)
        axes[i, 1].set_title("Reconstruction", fontsize=12)

plt.suptitle(f"{MODEL_NAME} on dSprites", fontsize=14)
plt.tight_layout()
plt.savefig(f"recon_{MODEL_NAME}.png", dpi=150, bbox_inches='tight')
plt.show()